In [ ]:
from pathlib import Path

# Find the repository root so these notebooks work on other computers.
PROJECT_ROOT = Path.cwd().resolve()
while not (PROJECT_ROOT / "Dataset Specs").is_dir() and PROJECT_ROOT != PROJECT_ROOT.parent:
    PROJECT_ROOT = PROJECT_ROOT.parent
if not (PROJECT_ROOT / "Dataset Specs").is_dir():
    raise FileNotFoundError("Start Jupyter from inside the Hospital Dataset project folder.")

import pandas as pd
import numpy as np
from pathlib import Path
from sklearn.metrics import mean_squared_error, mean_absolute_error

data_dir = Path(
    f"{PROJECT_ROOT}/"
    "synthetic_hospital_outputs/90_Days"
)

arima_df = pd.read_csv(
    data_dir / "arima_occupancy_predictions.csv",
    parse_dates=["date"]
)

rf_df = pd.read_csv(
    data_dir / "random_forest_occupancy_predictions.csv",
    parse_dates=["date"]
)

gb_df = pd.read_csv(
    data_dir / "gradient_boosting_occupancy_predictions.csv",
    parse_dates=["date"]
)

df_fusion = arima_df.merge(
    rf_df, on="date", suffixes=("_arima", "_rf")
)

df_fusion = df_fusion.merge(
    gb_df, on="date"
)

print(df_fusion.head())
print("Combined rows:", len(df_fusion))
print("Columns:", df_fusion.columns.tolist())

In [ ]:
print("ARIMA:", arima_df.columns.tolist())
print("Random Forest:", rf_df.columns.tolist())
print("Gradient Boosting:", gb_df.columns.tolist())
print("Merged:", df_fusion.columns.tolist())

In [ ]:
rf_cols = ["rf_1", "rf_2", "rf_3", "rf_4", "rf_5"]
gb_cols = ["gb_1", "gb_2", "gb_3", "gb_4", "gb_5"]

df_fusion["rf_predicted_occupied_beds"] = df_fusion[rf_cols].mean(axis=1)
df_fusion["gb_predicted_occupied_beds"] = df_fusion[gb_cols].mean(axis=1)

print(df_fusion[
    [
        "date",
        "actual_occupied_beds",
        "arima_predicted_occupied_beds",
        "rf_predicted_occupied_beds",
        "gb_predicted_occupied_beds"
    ]
].head())

print("Combined rows:", len(df_fusion))
print("Missing values:", df_fusion.isna().sum().sum())

In [ ]:
df_fusion["fusion_predicted_occupied_beds"] = df_fusion[
    [
        "arima_predicted_occupied_beds",
        "rf_predicted_occupied_beds",
        "gb_predicted_occupied_beds"
    ]
].mean(axis=1)

print(df_fusion[
    ["date", "actual_occupied_beds", "fusion_predicted_occupied_beds"]
].head())

In [ ]:
from sklearn.metrics import mean_squared_error, mean_absolute_error
import numpy as np
import pandas as pd

y_true = df_fusion["actual_occupied_beds"]

models = {
    "ARIMA": df_fusion["arima_predicted_occupied_beds"],
    "Random Forest": df_fusion["rf_predicted_occupied_beds"],
    "Gradient Boosting": df_fusion["gb_predicted_occupied_beds"],
    "Fusion Average": df_fusion["fusion_predicted_occupied_beds"]
}

results = []

for model_name, predictions in models.items():
    rmse = np.sqrt(mean_squared_error(y_true, predictions))
    mae = mean_absolute_error(y_true, predictions)
    mape = np.mean(np.abs((y_true - predictions) / y_true)) * 100

    results.append({
        "Model": model_name,
        "RMSE": rmse,
        "MAE": mae,
        "MAPE (%)": mape
    })

metrics_df = pd.DataFrame(results).round(3)
print(metrics_df)

In [ ]:
from pathlib import Path

output_dir = Path(
    f"{PROJECT_ROOT}/"
    "synthetic_hospital_outputs/90_Days"
)
output_dir.mkdir(parents=True, exist_ok=True)

# Save performance metrics
metrics_df.to_csv(
    output_dir / "fusion_baseline_metrics.csv",
    index=False
)

# Save actual values and all model predictions
df_fusion.to_csv(
    output_dir / "fusion_baseline_predictions.csv",
    index=False
)

print("Metrics saved:", output_dir / "fusion_baseline_metrics.csv")
print("Predictions saved:", output_dir / "fusion_baseline_predictions.csv")